# Glidelageret: Hvordan kan en tynn oljefilm bære en roterende aksel?

## Pilotprosjekt for Matematikk 1

En glidelagerbøssing omgir en roterende aksel. Mellom metallflatene ligger en svært tynn oljefilm. Når akselen roterer, trekkes olje inn i et stadig smalere gap. Det kan bygge seg opp et trykk som bærer akselen og holder metallflatene fra hverandre.

Prosjektet krever ikke forkunnskaper i tribologi eller fluidmekanikk. De mekaniske begrepene introduseres før de brukes, mens de mer avanserte modelligningene gis ferdig.

Prosjektet har tre hoveddeler:

1. **Oljekilen:** filmgeometri og en diskret trykkligning på formen $Ap=b$
2. **Oppvarming:** en skalar temperatur-ODE med og uten temperaturavhengig viskositet
3. **Akselbevegelse:** et koblet ODE-system for akselsenterets bane

### Læringsmål

Etter prosjektet skal du kunne

- beskrive klaring, eksentrisitet og minste filmtykkelse,
- bygge og løse et tridiagonalt lineært system,
- summere et diskret trykkfelt til en kraftvektor,
- løse en lineær førsteordens ODE analytisk og med Euler,
- tolke en enkel termisk energibalanse,
- skrive en andreordens mekanisk modell som et førsteordens system,
- bruke stivhets- og dempningsmatriser,
- gjennomføre et variabelbytte ved diagonalisering,
- simulere og visualisere akselsenterets bane.

### Modellavgrensning

Dette er en pedagogisk modell. Den kan brukes til å utforske sammenhenger og størrelsesordener, men ikke til å dimensjonere et virkelig lager eller dokumentere sikker drift.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Seks begreper vi trenger

## Aksel eller journal

Den roterende sylinderen i lageret.

## Lagerbøssing

Den stillestående sylindriske flaten rundt akselen.

## Radial klaring $C$

Forskjellen mellom lagerbøssingens indre radius og akselens radius.

## Eksentrisitet $e$

Avstanden mellom sentrum av akselen og sentrum av lagerbøssingen.

## Relativ eksentrisitet $\varepsilon$

$$
\varepsilon=\frac{e}{C},\qquad 0\leq\varepsilon<1.
$$

## Smørefilmtykkelse $h(\theta)$

Den lokale avstanden mellom aksel og bøssing. I vår modell er

$$
\boxed{h(\theta)=C\bigl(1+\varepsilon\cos\theta\bigr).}
$$

Dermed er

$$
h_{min}=C(1-\varepsilon),\qquad h_{max}=C(1+\varepsilon).
$$

# Del A: Hvordan kan oljen bære akselen?

## A.1 Filmgeometrien

Før vi beregner trykket, undersøker vi bare avstanden mellom metallflatene.

Bruk

$$R=50\ \text{mm},\qquad C=0.075\ \text{mm}.$$

Plott filmtykkelsen for flere verdier av $\varepsilon$.

In [ ]:
R = 50e-3
C = 0.075e-3

theta = np.linspace(0.0, 2*np.pi, 500)

def filmtykkelse(theta, epsilon):
    return ...

for epsilon in [0.0, 0.3, 0.6, 0.9]:
    h = filmtykkelse(theta, epsilon)
    plt.plot(np.rad2deg(theta), 1e6*h, label=f"epsilon={epsilon}")

plt.xlabel("Vinkel theta i grader")
plt.ylabel("Filmtykkelse i mikrometer")
plt.grid()
plt.legend()
plt.show()

## Oppgave A1: Tolk geometrien

1. Kontroller numerisk formelen for $h_{min}$.
2. Hva skjer med minste filmtykkelse når $\varepsilon$ nærmer seg 1?
3. Hvorfor øker faren for metallkontakt når $h_{min}$ blir svært liten?
4. Marker den delen av omkretsen der filmen blir smalere i valgt rotasjonsretning.

## A.2 Fra oljekile til trykk

Når akselen roterer, drar overflaten olje inn i det innsnevrende gapet. Oljen tvinges gjennom en stadig tynnere film, og det kan bygges opp trykk.

En tynnfilmtilnærming gir den endimensjonale Reynolds-ligningen

$$
\frac{d}{d\theta}
\left(h^3\frac{dp}{d\theta}\right)
=
6\mu R^2\Omega\frac{dh}{d\theta}.
$$

Her er

- $p(\theta)$ oljefilmtrykket,
- $\mu$ oljens dynamiske viskositet,
- $R$ akselradiusen,
- $\Omega$ akselens vinkelhastighet.

Du skal ikke utlede ligningen. Hovedoppgaven er å gjøre den om til et lineært ligningssystem.

Vi løser bare på den aktive delen $0\leq\theta\leq\pi$ og bruker de pedagogiske randbetingelsene

$$p(0)=p(\pi)=0.$$

I resten av lageret setter vi trykket lik omgivelsestrykket. Dette er en enkel måte å unngå ikke-fysiske negative trykk på.

## A.3 Fra differensialligning til fem ukjente trykk

Dette er prosjektets mest tekniske overgang. Derfor bruker vi først et **fast og lite rutenett**.

Vi deler den aktive halvsirkelen i seks like intervaller:

$$
N=6,\qquad \Delta\theta=\frac{\pi}{6}.
$$

Punktene er

$$
\theta_0=0,\ \theta_1=\frac{\pi}{6},\ldots,
\theta_6=\pi.
$$

Trykket i endepunktene er kjent:

$$p_0=0,\qquad p_6=0.$$

Dermed har vi bare fem ukjente:

$$
\boxed{p_1,p_2,p_3,p_4,p_5.}
$$

Hovedideen med diskretisering er å erstatte én funksjon $p(\theta)$ med et lite antall trykkverdier. Differensialligningen erstattes så av én algebraisk ligning i hvert indre punkt.

Vi innfører koeffisientene

$$
a_{j-1/2}=\frac{h_{j-1/2}^3}{\Delta\theta^2},
\qquad
a_{j+1/2}=\frac{h_{j+1/2}^3}{\Delta\theta^2},
$$

og en kjent høyreside

$$
b_j=6\mu R^2\Omega
\frac{h_{j+1}-h_{j-1}}{2\Delta\theta}.
$$

Da blir ligningen i indre punkt $j$

$$
\boxed{
a_{j-1/2}p_{j-1}
-\bigl(a_{j-1/2}+a_{j+1/2}\bigr)p_j
+a_{j+1/2}p_{j+1}=b_j.
}
$$

Hver ligning bruker bare trykket i punktet selv og de to nærmeste nabopunktene. Derfor får matrisen bare ikke-nullverdier på tre diagonaler.

## Oppgave A2: Skriv ut de fem ligningene

Se først på ligningen for $j=1$:

$$
a_{1/2}p_0-(a_{1/2}+a_{3/2})p_1+a_{3/2}p_2=b_1.
$$

Siden $p_0=0$, blir den

$$
-(a_{1/2}+a_{3/2})p_1+a_{3/2}p_2=b_1.
$$

For $j=3$ får vi

$$
a_{5/2}p_2-(a_{5/2}+a_{7/2})p_3+a_{7/2}p_4=b_3.
$$

Den siste ligningen bruker $p_6=0$ på tilsvarende måte.

Koden nedenfor beregner koeffisientene og skriver ut én rad om gangen. Kjør cellen og sammenlign utskriften med ligningene over. Du trenger ikke programmere en generell matrisebyggingsalgoritme i hovedoppgaven.

In [ ]:
mu = 0.015       # Pa s
Omega = 2*np.pi*50  # 50 omdreininger per sekund
L = 0.10            # lagerbredde i meter
epsilon = 0.65
N_trykk = 6          # fast rutenett i hovedoppgaven

theta_A = np.linspace(0.0, np.pi, N_trykk + 1)
dtheta = theta_A[1] - theta_A[0]
h_A = filmtykkelse(theta_A, epsilon)
h_halv = 0.5*(h_A[:-1] + h_A[1:])

A_trykk = np.zeros((5, 5))
b_trykk = np.zeros(5)

for j in range(1, N_trykk):
    rad = j - 1
    a_minus = h_halv[j - 1]**3 / dtheta**2
    a_plus = h_halv[j]**3 / dtheta**2

    A_trykk[rad, rad] = -(a_minus + a_plus)

    if j > 1:
        A_trykk[rad, rad - 1] = a_minus

    if j < N_trykk - 1:
        A_trykk[rad, rad + 1] = a_plus

    b_trykk[rad] = (
        6*mu*R**2*Omega
        * (h_A[j + 1] - h_A[j - 1])
        / (2*dtheta)
    )

np.set_printoptions(precision=4, suppress=False)
print("Vinkelpunkter i grader:", np.rad2deg(theta_A))
print("Filmtykkelser i mikrometer:", 1e6*h_A)
print("
Matrise A:
", A_trykk)
print("
Høyreside b:
", b_trykk)

## Oppgave A3: Les matrisen som fem ligninger

Matriselikningen er

$$
A
\begin{pmatrix}
p_1\\p_2\\p_3\\p_4\\p_5
\end{pmatrix}
=b.
$$

Gjør følgende før du løser systemet:

1. Skriv første matriserad som en ligning i $p_1$ og $p_2$.
2. Skriv tredje matriserad som en ligning i $p_2,p_3,p_4$.
3. Skriv siste matriserad som en ligning i $p_4,p_5$.
4. Forklar hvorfor første og siste rad bare har to trykkukjente.
5. Forklar hvorfor matrisen er tridiagonal.

Løs deretter systemet. Sett de fem ukjente trykkene inn mellom de kjente randverdiene $p_0=p_6=0$.

In [ ]:
p_indre = np.linalg.solve(A_trykk, b_trykk)

p_A = np.zeros(N_trykk + 1)
p_A[1:-1] = p_indre

residual = A_trykk @ p_indre - b_trykk

print("Indre trykkverdier i MPa:", p_indre/1e6)
print("Residualnorm:", np.linalg.norm(residual))
print("Maksimalt trykk:", np.max(p_A)/1e6, "MPa")

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(np.rad2deg(theta_A), 1e6*h_A, "o-")
ax[0].set_ylabel("Filmtykkelse i mikrometer")
ax[0].grid()

ax[1].plot(np.rad2deg(theta_A), p_A/1e6, "o-")
ax[1].set_xlabel("Vinkel theta i grader")
ax[1].set_ylabel("Trykk i MPa")
ax[1].grid()

plt.show()

## A.4 Fra seks trykkintervaller til en kraft

Trykket virker på den krumme akseloverflaten. Vi deler derfor overflaten inn i de samme seks små vinkelintervallene.

For ett lite intervall rundt vinkelen $\theta_j$ er arealet omtrent

$$
\Delta A\approx RL\Delta\theta.
$$

Trykk ganger areal gir kraft. Kraftbidragets størrelse er derfor omtrent

$$
\Delta F_j\approx p_jRL\Delta\theta.
$$

Kraften peker normalt på akseloverflaten. En normalvektor ved vinkelen $\theta_j$ er

$$
n_j=
\begin{pmatrix}
\cos\theta_j\\
\sin\theta_j
\end{pmatrix}.
$$

Dermed blir kraftbidraget

$$
\Delta\mathbf F_j
\approx
p_jRL\Delta\theta
\begin{pmatrix}
\cos\theta_j\\
\sin\theta_j
\end{pmatrix}.
$$

Når alle små bidrag summeres, får vi

$$
F_x\approx RL\sum_jp_j\cos\theta_j\Delta\theta,
$$

$$
F_y\approx RL\sum_jp_j\sin\theta_j\Delta\theta.
$$

Dette er bare en vektorsum: hvert trykkpunkt bidrar med en liten kraft i sin lokale normalretning.

## Oppgave A4: Summer kraftbidragene

1. Beregn kraftbidraget fra hvert av de fem indre punktene.
2. Skriv ut bidragene som vektorer.
3. Summer dem til en samlet kraft.
4. Beregn kraftens størrelse og retning.
5. Lag et enkelt plott av de fem bidragsvektorene dersom du ønsker en visuell kontroll.

Fortegnet avhenger av om normalvektoren velges utover fra akselen eller som kraftretningen på akselen. I denne piloten er vi mest opptatt av kraftens størrelse og hvordan den endres med parametrene.

In [ ]:
theta_indre = theta_A[1:-1]

kraftbidrag = (
    p_indre[:, None]
    * R*L*dtheta
    * np.column_stack([np.cos(theta_indre), np.sin(theta_indre)])
)

for j, bidrag in enumerate(kraftbidrag, start=1):
    print(f"Punkt {j}: dF = {bidrag} N")

F_vektor = np.sum(kraftbidrag, axis=0)
F_x, F_y = F_vektor
F_lager = np.linalg.norm(F_vektor)
vinkel_kraft = np.arctan2(F_y, F_x)

print("
Samlet kraftvektor:", F_vektor, "N")
print("Kraftstørrelse:", F_lager, "N")
print("Kraftretning:", np.rad2deg(vinkel_kraft), "grader")

## Valgfri fordypning: Flere rutenettpunkter

Hovedoppgaven bruker $N=6$ for at alle fem ligningene skal være synlige. Når strukturen er forstått, kan samme mønster brukes for et større antall punkter.

I en generell kode blir rad $j$ fylt med

- én koeffisient for $p_{j-1}$,
- én koeffisient for $p_j$,
- én koeffisient for $p_{j+1}$.

Første og siste rad mangler én nabo fordi randtrykkene allerede er kjent. En ferdig generell matrisefunksjon kan gis av lærer dersom et finere trykkplott er ønskelig. Den generelle programmeringen er ikke et læringsmål i hovedprosjektet.

## Oppgave A4: Parameterstudie

Varier én parameter om gangen:

- relativ eksentrisitet $\varepsilon$,
- viskositet $\mu$,
- vinkelhastighet $\Omega$.

For hvert tilfelle beregner du

- minste filmtykkelse,
- maksimalt trykk,
- samlet lagerkraft.

Forklar hvilke parametre som øker bæreevnen, og hvorfor et stort trykk ikke nødvendigvis betyr at lageret har en trygg filmtykkelse.

In [ ]:
epsilon_verdier = np.linspace(0.1, 0.9, 17)
resultater = []

for eps in epsilon_verdier:
    # Bygg og løs trykksystemet, og lagre h_min, p_max og kraft.
    pass

# Lag relevante plott.

## Oppgave A5: Likevekt under en gitt last

En forenklet designoppgave er å finne den eksentrisiteten som gir omtrent samme lagerkraft som en kjent ytre last $W$.

Beregn lagerkraften på et fint rutenett av $\varepsilon$-verdier. Finn den verdien som gir minst

$$|F_{lager}(\varepsilon)-W|.$$

Kontroller også minste filmtykkelse ved denne eksentrisiteten.

In [ ]:
W = 50_000.0  # N

# Finn omtrent riktig epsilon ved et rutenettsøk.

# Del B: Hvor varm blir oljen?

Rotasjonen gir viskøs friksjon. Mekanisk energi omdannes til varme, mens varme samtidig forlater lageret gjennom lagerhuset og den gjennomstrømmende oljen.

Vi bruker en samlet temperatur $T(t)$ for lager og olje. Dette er en grov modell, men den gir en tydelig energibalanse.

## Begreper

- **Viskositet $\mu$:** hvor sterkt oljen motsetter seg skjærbevegelse
- **Friksjonseffekt $P_f$:** mekanisk energi som går over til varme per sekund
- **Termisk kapasitet $C_T$:** energi som kreves for å øke temperaturen
- **Kjølekoeffisient $k_T$:** hvor raskt varme avgis når temperaturen er over omgivelsestemperaturen

## B.1 Lineær termisk modell

Vi begynner med konstant varmeproduksjon:

$$
\boxed{C_T\dot T=P_0-k_T(T-T_{omg}).}
$$

Likevektstemperaturen er

$$
T^*=T_{omg}+\frac{P_0}{k_T},
$$

og tidskonstanten er

$$
\tau_T=\frac{C_T}{k_T}.
$$

## Oppgave B1: Analytisk løsning

Vis at løsningen er

$$
T(t)=T^*+(T_0-T^*)e^{-t/\tau_T}.
$$

Bruk

$$C_T=8.0\cdot10^4\ \mathrm{J/K},\quad
P_0=1800\ \mathrm W,$$

$$k_T=75\ \mathrm{W/K},\quad
T_{omg}=20^\circ\mathrm C,\quad T_0=20^\circ\mathrm C.$$

In [ ]:
C_T = 8.0e4
P0 = 1800.0
k_T = 75.0
T_omg = 20.0
T0 = 20.0

T_likevekt = ...
tau_T = ...

print("Likevektstemperatur:", T_likevekt, "grader C")
print("Termisk tidskonstant:", tau_T, "s")

## Oppgave B2: Euler og eksakt løsning

Implementer Euler, og sammenlign med håndløsningen over minst fem tidskonstanter.

In [ ]:
def temperatur_lineær(t, T):
    return ...


def euler_skalar(f, y0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    y = np.zeros(n + 1)
    y[0] = y0

    for k in range(n):
        y[k + 1] = ...

    return t, y


t_B, T_B = euler_skalar(temperatur_lineær, T0, 5*tau_T, tau_T/100)
T_eksakt = ...

# Plott og beregn maksimal feil.

## B.2 Temperaturavhengig viskositet

Olje blir vanligvis mindre viskøs når temperaturen øker. Vi bruker den enkle modellen

$$
\mu(T)=\mu_0e^{-\beta(T-T_{ref})}.
$$

Vi antar videre at friksjonseffekten er omtrent proporsjonal med viskositeten:

$$P_f(T)=K_f\mu(T)\Omega^2.$$

Da får vi den ikke-lineære ODE-en

$$
\boxed{
C_T\dot T=K_f\mu_0e^{-\beta(T-T_{ref})}\Omega^2
-k_T(T-T_{omg}).
}
$$

## Oppgave B3: Ikke-lineær temperaturmodell

Velg $K_f$ slik at varmeproduksjonen er $P_0$ ved referansetemperaturen. Simuler med Euler og sammenlign med den lineære modellen.

Undersøk hvordan slutt-temperaturen påvirkes av $\beta$ og $\Omega$.

In [ ]:
mu0 = 0.015
beta = 0.025
T_ref = 20.0
Omega_B = Omega
K_f = P0/(mu0*Omega_B**2)


def viskositet(T):
    return ...


def temperatur_ikke_lineær(t, T):
    P_f = ...
    return ...

# Simuler, plott og sammenlign.

## Oppgave B4: Koble tilbake til trykkmodellen

Bruk temperaturen fra den ikke-lineære modellen til å beregne $\mu(T)$. Løs deretter trykkproblemet fra del A med

1. kald olje,
2. varm olje.

Sammenlign maksimalt trykk og samlet bærekraft ved samme geometri og rotasjonshastighet.

Forklar hvorfor temperaturkontroll er viktig selv om lavere viskositet kan redusere friksjonstapet.

# Del C: Hvordan beveger akselen seg?

Vi studerer små bevegelser av akselsenteret rundt en statisk likevekt.

Først i én retning:

$$m\ddot x+c\dot x+kx=F(t).$$

Dette kan tolkes som

- $m\ddot x$: akselens treghet,
- $c\dot x$: oljefilm som demper bevegelsen,
- $kx$: oljefilm som prøver å føre akselen tilbake,
- $F(t)$: ytre kraft.

I et virkelig glidelager kan en forskyvning i én retning også gi en oljekraft i den andre retningen. Derfor bruker vi senere matriser.

## C.1 To uavhengige retninger

La

$$q=\begin{pmatrix}x\\y\end{pmatrix}.$$

Den enkleste modellen er

$$m\ddot q+c\dot q+Kq=f(t),$$

med diagonal stivhetsmatrise

$$K=\begin{pmatrix}k_x&0\\0&k_y\end{pmatrix}.$$

De to retningene kan da analyseres uavhengig.

## Oppgave C1: Normalfrekvenser

Bruk

$$m=75\ \mathrm{kg},\qquad
k_x=1.8\cdot10^6\ \mathrm{N/m},$$

$$k_y=2.4\cdot10^6\ \mathrm{N/m},\qquad
c=8.0\cdot10^3\ \mathrm{N\,s/m}.$$

Beregn de udempede naturlige vinkelfrekvensene

$$\omega_x=\sqrt{k_x/m},\qquad
\omega_y=\sqrt{k_y/m}.$$

In [ ]:
m_rotor = 75.0
kx = 1.8e6
ky = 2.4e6
c_iso = 8.0e3

omega_x = ...
omega_y = ...

print("Naturlige frekvenser i Hz:", omega_x/(2*np.pi), omega_y/(2*np.pi))

## C.2 Krysskoblet oljefilm

I den mer generelle modellen er

$$
K=\begin{pmatrix}
k_{xx}&k_{xy}\\
k_{yx}&k_{yy}
\end{pmatrix},
\qquad
D=\begin{pmatrix}
c_{xx}&c_{xy}\\
c_{yx}&c_{yy}
\end{pmatrix}.
$$

Modellen er

$$
\boxed{m\ddot q+D\dot q+Kq=f(t).}
$$

Kryssleddet $k_{xy}$ betyr for eksempel at en forskyvning i $y$ kan gi en kraftkomponent i $x$.

I hovedprosjektet får du $K$ og $D$ som modellparametre. Å beregne dem direkte fra Reynolds-ligningen er en mer avansert oppgave.

## C.3 Førsteordensform

Sett

$$v=\dot q$$

og bruk tilstanden

$$X=(x,y,v_x,v_y)^T.$$

Da blir

$$
\begin{aligned}
\dot x&=v_x,\\
\dot y&=v_y,\\
m\dot v_x&=-k_{xx}x-k_{xy}y-c_{xx}v_x-c_{xy}v_y+F_x(t),\\
m\dot v_y&=-k_{yx}x-k_{yy}y-c_{yx}v_x-c_{yy}v_y+F_y(t).
\end{aligned}
$$

## Oppgave C2: Fri bevegelse

Bruk de symmetriske matrisene

$$
K=10^6\begin{pmatrix}2.0&0.45\\0.45&1.6\end{pmatrix},
$$

$$
D=10^4\begin{pmatrix}1.8&0.20\\0.20&1.4\end{pmatrix}.
$$

Slipp akselen fra en liten forskyvning og simuler uten ytre kraft.

In [ ]:
K = 1e6*np.array([
    [2.0, 0.45],
    [0.45, 1.6]
])

D = 1e4*np.array([
    [1.8, 0.20],
    [0.20, 1.4]
])


def rotor_fri(t, X):
    q = X[:2]
    v = X[2:]
    dq = v
    dv = ...
    return np.concatenate([dq, dv])


def euler_system(f, X0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(X0)))
    X[0] = X0

    for k in range(n):
        X[k + 1] = ...

    return t, X


X0 = np.array([20e-6, -10e-6, 0.0, 0.0])
t_C, X_C = euler_system(rotor_fri, X0, sluttid=0.5, h=2e-5)

x_C = X_C[:, 0]
y_C = X_C[:, 1]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(t_C, 1e6*x_C, label="x")
ax[0].plot(t_C, 1e6*y_C, label="y")
ax[0].set_xlabel("Tid s")
ax[0].set_ylabel("Forskyvning i mikrometer")
ax[0].legend()
ax[0].grid()

ax[1].plot(1e6*x_C, 1e6*y_C)
ax[1].set_xlabel("x i mikrometer")
ax[1].set_ylabel("y i mikrometer")
ax[1].axis("equal")
ax[1].grid()

plt.show()

## Oppgave C3: Diagonalisering og normale moder

Siden $K$ er symmetrisk, kan den diagonaliseres med ortogonale egenvektorer:

$$K=P\Lambda P^T.$$

Gjør variabelbyttet

$$q=Pz.$$

Dersom dempingsmatrisen også er diagonal i den samme basisen, blir de nye koordinatene to uavhengige moder.

1. Finn egenverdier og egenvektorer til $K$.
2. Kontroller $P^TP=I$.
3. Transformér startforskyvningen til modalkoordinater.
4. Tolk egenvektorene som bevegelsesretninger i lagerplanet.

In [ ]:
eg_k, P = ...
Lambda = ...

print("Egenverdier:", eg_k)
print("Egenvektorer:
", P)
print("Kontroll P^T P:
", ...)
print("Kontroll K = P Lambda P^T:
", ...)

q0 = X0[:2]
z0 = ...
print("Startforskyvning i modalkoordinater:", z0)

## C.4 Ubalanse i en roterende aksel

Dersom rotorens massesenter ligger litt utenfor rotasjonsaksen, oppstår en roterende ubalansekraft. Vi modellerer den som

$$
f(t)=m_ue_u\Omega_r^2
\begin{pmatrix}
\cos\Omega_rt\\
\sin\Omega_rt
\end{pmatrix}.
$$

Her er

- $m_u$ en ubalansemasse,
- $e_u$ avstanden fra rotasjonsaksen,
- $\Omega_r$ rotorens vinkelhastighet.

## Oppgave C4: Tvungen akselbane

Implementer ubalansekraften og simuler ved flere rotasjonshastigheter. Plott akselbanen etter at den første transienten har avtatt.

In [ ]:
m_u = 0.020
e_u = 0.0005
Omega_r = 2*np.pi*40


def ubalansekraft(t, Omega_r=Omega_r):
    amplitude = m_u*e_u*Omega_r**2
    return amplitude*np.array([np.cos(Omega_r*t), np.sin(Omega_r*t)])


def rotor_med_ubalanse(t, X):
    q = X[:2]
    v = X[2:]
    dq = v
    dv = ...
    return np.concatenate([dq, dv])

# Simuler, forkast den første transienten og plott akselbanen.

## Oppgave C5: Hastighetsstudie

Kjør modellen for flere rotorhastigheter. Registrer maksimal baneamplitude etter transienten.

Sammenlign rotasjonshastighetene med de naturlige frekvensene fra stivhetsmatrisen. Forklar hvorfor responsen kan bli stor når eksitasjonsfrekvensen ligger nær en naturlig frekvens.

## Oppgave C6: Minste filmtykkelse under bevegelse

I en svært enkel geometrisk kontroll kan akselsenterets avstand fra lagerets sentrum beregnes som

$$e(t)=\sqrt{x(t)^2+y(t)^2}.
$$

Da er

$$h_{min}(t)\approx C-e(t).$$

Beregn minste verdi av $h_{min}(t)$ i simuleringen. Stopp eller marker modellen dersom $e\geq C$.

Forklar hvorfor denne kontrollen bare er geometrisk. Den dynamiske ODE-modellen er linearisert rundt en likevekt og blir ikke pålitelig ved forskyvninger som nærmer seg hele klaringen.

In [ ]:
e_t = ...
h_min_t = ...

print("Minste beregnede filmtykkelse:", np.min(h_min_t)*1e6, "mikrometer")

# Fordypning: Fra trykkfelt til lagerkoeffisienter

I en mer avansert modell kan oljefilmkreften skrives som en funksjon

$$F:\mathbb R^2\to\mathbb R^2,
\qquad
F(x,y)=\begin{pmatrix}F_x(x,y)\\F_y(x,y)\end{pmatrix}.
$$

Stivhetsmatrisen nær en likevekt kan estimeres fra den negative Jacobimatrisen:

$$K\approx-\frac{\partial F}{\partial q}.
$$

Et element kan for eksempel tilnærmes med sentral differanse:

$$
\frac{\partial F_x}{\partial x}
\approx
\frac{F_x(x_0+\delta,y_0)-F_x(x_0-\delta,y_0)}{2\delta}.
$$

Dette knytter trykkmodellen i del A til dynamikken i del C, men passer bedre som videreføring når studentene har lært funksjoner fra $\mathbb R^n$ til $\mathbb R^m$ og partiellderiverte.

# Modellkritikk

Diskuter minst fem punkter:

- Lageret behandles som svært langt i trykkmodellen.
- Trykket løses bare i en valgt aktiv halvdel av filmen.
- Filmruptur og kavitasjon behandles med en enkel trykkavkutting.
- Viskositeten antas romlig uniform.
- Trykkmodellen er stasjonær, mens akseldynamikken er tidsavhengig.
- Temperaturmodellen bruker én samlet temperatur.
- Stivhets- og dempningsmatrisene er gitt, ikke utledet fra trykkfeltet.
- Rotoren modelleres som en stiv masse i ett lagerplan.
- Deformasjon av aksel og lagerhus er utelatt.
- Slitasje, ruhet og metallkontakt er utelatt.
- Smøremiddeltilførsel og lekkasje behandles ikke detaljert.
- Euler-metoden kan kreve liten steglengde ved høye naturlige frekvenser.

## Mulig praktisk videreføring

Hvis et egnet undervisningslager blir tilgjengelig, kan modellen senere sammenlignes med målinger av for eksempel

- lager- eller oljetemperatur,
- friksjonsmoment,
- akselposisjon eller vibrasjon,
- rotasjonshastighet,
- påført last.

Et praktisk forsøk må planlegges og risikovurderes av fagmiljøet. Denne notebooken forutsetter ikke at et slikt forsøk gjennomføres.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hva klaring, eksentrisitet og filmtykkelse betyr,
2. hvorfor en roterende aksel kan bygge opp trykk i en konvergerende oljefilm,
3. hvordan den diskrete trykkligningen ga et lineært system,
4. hvordan trykkverdiene ble summert til en kraftvektor,
5. hvordan varmeproduksjon og kjøling ga en temperatur-ODE,
6. hvordan temperatur påvirket viskositet og bæreevne,
7. hvordan den todimensjonale akselmodellen ble skrevet som et førsteordenssystem,
8. hva egenvektorene til stivhetsmatrisen betyr,
9. hvordan ubalanse påvirket akselbanen,
10. hvilke modellforutsetninger som er viktigst.

## Referanse for prosjektutviklingen

Prosjektets statiske lagerdel er inspirert av kapitlet *Static Characteristics of Journal Bearings* i det vedlagte materialet om hydrodynamisk smøring.

Studentene trenger ikke lese bokkapitlet for å gjennomføre prosjektet.